# Day 30 Tutorial：PyG `Data` 的字段检查

## Goal

把 Day 29 的同一张四节点无向图装入 `torch_geometric.data.Data`，严格检查 `x`、`edge_index`、`y`、`num_nodes`、dtype、节点编号范围和无向边双向性。本教程不定义或训练模型。

## Setup

需要可导入 `torch` 与 `torch_geometric` 的独立 GNN 环境。标签 `0.75` 和节点特征都是教学值，不来自实验，也不具有真实化学语义。

In [1]:
import warnings

warnings.filterwarnings('ignore', message='IProgress not found.*')

import torch
import torch_geometric
from torch_geometric.data import Data

SEED = 30
torch.manual_seed(SEED)

print('torch version:', torch.__version__)
print('torch_geometric version:', torch_geometric.__version__)

torch version: 2.13.0
torch_geometric version: 2.8.0.post1


## Steps

`x` 每个节点一行，使用 `float32`；`edge_index` 每列一个方向记录，使用可索引的 `torch.long`；`y` 是一张图的一个连续教学标签。四条无向边展开为八个方向记录。

In [2]:
x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
        [0.5, 0.5],
    ],
    dtype=torch.float32,
)

undirected_edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
directed_edges = []
for source, target in undirected_edges:
    directed_edges.append((source, target))
    directed_edges.append((target, source))

edge_index = torch.tensor(directed_edges, dtype=torch.long).T.contiguous()
y = torch.tensor([0.75], dtype=torch.float32)
graph = Data(x=x, edge_index=edge_index, y=y)

print(graph)
print('x:', tuple(graph.x.shape), graph.x.dtype)
print('edge_index:', tuple(graph.edge_index.shape), graph.edge_index.dtype)
print('y:', tuple(graph.y.shape), graph.y.dtype)
print('num_nodes:', graph.num_nodes)

Data(x=[4, 2], edge_index=[2, 8], y=[1])
x: (4, 2) torch.float32
edge_index: (2, 8) torch.int64
y: (1,) torch.float32
num_nodes: 4


逐列读取 `edge_index`。先转置成 `(8, 2)`，每一行就变成一对“来源、目标”，便于检查反方向是否存在。

In [3]:
directed_pairs = {
    (int(source), int(target))
    for source, target in graph.edge_index.T.tolist()
}

print('方向记录数:', len(directed_pairs))
print('排序后的方向记录:', sorted(directed_pairs))

方向记录数: 8
排序后的方向记录: [(0, 1), (0, 3), (1, 0), (1, 2), (2, 1), (2, 3), (3, 0), (3, 2)]


## Checks

以下断言分别验证字段存在、shape、dtype、节点数、编号范围、没有重复方向、无自环以及每条方向记录都有反方向。

In [4]:
assert graph.x is not None
assert graph.edge_index is not None
assert graph.y is not None

assert tuple(graph.x.shape) == (4, 2)
assert graph.x.dtype == torch.float32
assert tuple(graph.edge_index.shape) == (2, 8)
assert graph.edge_index.dtype == torch.long
assert tuple(graph.y.shape) == (1,)
assert graph.y.dtype == torch.float32
assert graph.num_nodes == 4

assert int(graph.edge_index.min()) >= 0
assert int(graph.edge_index.max()) < graph.num_nodes
assert len(directed_pairs) == graph.edge_index.shape[1]

for source, target in directed_pairs:
    assert source != target
    assert (target, source) in directed_pairs

assert graph.validate(raise_on_error=True)
print('全部 Data 结构检查通过。')
print('边界：Data 只是容器，本教程没有训练模型。')

全部 Data 结构检查通过。
边界：Data 只是容器，本教程没有训练模型。


## Next Steps

关闭教程，独立完成 `03_exercises.md`。只有能够不看答案解释所有字段与检查后，才进入 Day 31 的第一个 GCN；课程输出不算个人学习证据或材料研究结果。